# Cafe Sales Dataset — Data Cleaning & Quality Assessment

**Objective:** Convert `dirty_cafe_sales.csv` into a documented, analysis-ready CSV while preserving a reproducible audit trail.

**Approach:** inspect first, document decisions, clean a copy, validate the result, compare before/after metrics, and export to a *new* file. Run cells from top to bottom.

> **Important:** This notebook uses transparent rules rather than silently guessing. For example, invalid numeric placeholders (`ERROR`, `UNKNOWN`) become missing values; transaction dates that cannot be parsed are excluded because a date cannot be reliably inferred from other rows. Review the final audit and business assumptions before using the data for reporting.

## 1. Import libraries and load the raw dataset

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# If the CSV is in the same folder as this notebook, this path works.
# The second path supports the provided notebook execution environment.
candidate_paths = [Path("dirty_cafe_sales.csv"), Path("/mnt/data/dirty_cafe_sales.csv")]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find dirty_cafe_sales.csv. Put it beside this notebook "
        "or update DATA_PATH to the CSV location."
    )

raw = pd.read_csv(DATA_PATH, dtype="string")
df = raw.copy(deep=True)

print(f"Loaded: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())
display(df.sample(min(5, len(df)), random_state=42))

Loaded: /mnt/data/dirty_cafe_sales.csv
Shape: 10,000 rows × 8 columns


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
6252,TXN_2919952,Juice,<NA>,3.0,15.0,Digital Wallet,In-store,2023-02-17
4684,TXN_4265056,Juice,1,3.0,3.0,Cash,<NA>,2023-07-16
1731,TXN_2463115,Sandwich,5,4.0,20.0,Credit Card,Takeaway,2023-09-12
4742,TXN_7619641,Juice,3,3.0,9.0,<NA>,In-store,2023-04-02
4521,TXN_9566593,Cake,2,3.0,6.0,Digital Wallet,<NA>,2023-09-30


## 2. Data quality report — before cleaning
This report treats common text placeholders such as `ERROR` and `UNKNOWN` as missing-like values, in addition to true nulls. It also checks exact duplicate rows, parse failures in numeric/date fields, and numeric IQR outliers.

In [2]:
PLACEHOLDERS = {"", "ERROR", "UNKNOWN", "N/A", "NA", "NULL", "NONE", "NAN"}

def missing_like_mask(series):
    text = series.astype("string").str.strip()
    return series.isna() | text.str.upper().isin(PLACEHOLDERS)

def make_quality_report(data):
    report = pd.DataFrame(index=data.columns)
    report["dtype_before"] = data.dtypes.astype(str)
    report["true_nulls"] = data.isna().sum()
    report["placeholder_values"] = {
        col: int((data[col].notna() & data[col].astype("string").str.strip().str.upper().isin(PLACEHOLDERS)).sum())
        for col in data.columns
    }
    report["missing_like_total"] = {
        col: int(missing_like_mask(data[col]).sum()) for col in data.columns
    }
    return report

quality_before = make_quality_report(raw)
display(quality_before)
print("Exact duplicate rows:", int(raw.duplicated().sum()))
print("Duplicate transaction IDs:", int(raw["Transaction ID"].duplicated().sum()) if "Transaction ID" in raw else "N/A")

,dtype_before,true_nulls,placeholder_values,missing_like_total
Transaction ID,string,0,0,0
Item,string,333,636,969
Quantity,string,138,341,479
Price Per Unit,string,179,354,533
Total Spent,string,173,329,502
Payment Method,string,2579,599,3178
Location,string,3265,696,3961
Transaction Date,string,159,301,460


Exact duplicate rows: 0
Duplicate transaction IDs: 0


In [3]:
# Parse diagnostics: how many values cannot be converted to the expected type?
numeric_cols = ["Quantity", "Price Per Unit", "Total Spent"]
for col in numeric_cols:
    parsed = pd.to_numeric(
        raw[col].astype("string").str.strip().mask(missing_like_mask(raw[col])),
        errors="coerce"
    )
    bad = raw[col].notna() & ~missing_like_mask(raw[col]) & parsed.isna()
    print(f"{col}: non-placeholder values failing numeric parse = {int(bad.sum())}")

date_col = "Transaction Date"
parsed_dates = pd.to_datetime(
    raw[date_col].astype("string").str.strip().mask(missing_like_mask(raw[date_col])),
    errors="coerce",
    format="mixed"
)
date_bad = raw[date_col].notna() & ~missing_like_mask(raw[date_col]) & parsed_dates.isna()
print(f"{date_col}: non-placeholder values failing date parse = {int(date_bad.sum())}")

Quantity: non-placeholder values failing numeric parse = 0
Price Per Unit: non-placeholder values failing numeric parse = 0
Total Spent: non-placeholder values failing numeric parse = 0
Transaction Date: non-placeholder values failing date parse = 0


In [4]:
# IQR outlier scan on numeric fields after coercing invalid strings to NaN.
# This is diagnostic only; outliers are not automatically deleted.
numeric_preview = pd.DataFrame({
    col: pd.to_numeric(
        raw[col].astype("string").str.strip().mask(missing_like_mask(raw[col])),
        errors="coerce"
    ) for col in numeric_cols
})
outlier_rows = []
for col in numeric_cols:
    s = numeric_preview[col].dropna()
    if s.empty:
        outlier_rows.append({"column": col, "Q1": np.nan, "Q3": np.nan,
                             "IQR": np.nan, "lower_bound": np.nan,
                             "upper_bound": np.nan, "outlier_count": 0})
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_rows.append({
        "column": col, "Q1": q1, "Q3": q3, "IQR": iqr,
        "lower_bound": lower, "upper_bound": upper,
        "outlier_count": int(((s < lower) | (s > upper)).sum())
    })
outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

,column,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count
0,Quantity,2.0,4.0,2.0,-1.0,7.0,0
1,Price Per Unit,2.0,4.0,2.0,-1.0,7.0,0
2,Total Spent,4.0,12.0,8.0,-8.0,24.0,259


## 3. Cleaning decisions (documented)

| Column(s) | Rule | Why |
|---|---|---|
| `Transaction ID` | Trim whitespace; store as pandas `string`; do not impute IDs | An identifier is not a measurable quantity. Inventing an ID can create false records. |
| `Item`, `Payment Method`, `Location` | Convert placeholder strings to missing; trim/case-normalise known categories; fill remaining missing values with the column mode | These are categorical fields. Mode is a simple baseline when the project requires a complete table, but it can distort category frequencies; missingness is reported before filling. |
| `Quantity`, `Price Per Unit` | Convert to numeric; placeholders and parse failures become missing; fill missing values with the median | Median is less sensitive to extreme values than mean and is a reasonable baseline for skewed transaction data. For real business use, validate against source receipts instead. |
| `Total Spent` | Convert to numeric; where quantity and unit price are available, calculate `Quantity × Price Per Unit`; use the median only when it still cannot be calculated | This field has a business rule that can recover values more reliably than statistical imputation. We do not overwrite an existing value unless it is missing/invalid. |
| `Transaction Date` | Parse to datetime; unparseable or placeholder dates become `NaT`; drop rows with no valid transaction date | Forward-filling dates could assign a transaction to the wrong day/month and bias time-series analysis. Date is essential for a dated sales record, so rows without it are excluded and counted. |
| Exact duplicate rows | Remove exact duplicates | Repeated copies can inflate sales totals. Transaction IDs are also checked, but rows with the same ID are not automatically deleted unless the entire row is identical. |
| Numeric IQR outliers | Flag and retain unless clearly impossible based on domain rules | A high quantity or spend may be a genuine bulk order. The IQR rule detects unusual values; it does not prove an error. This dataset's valid business bounds should be confirmed with the data owner. |

**Assumption:** this notebook prioritises a complete, practical teaching dataset. Mode/median imputation is explicitly flagged as a limitation and should not replace source-system correction for production finance data.

## 4. Clean the data

In [5]:
clean = raw.copy(deep=True)
rows_before = len(clean)
duplicates_before = int(clean.duplicated().sum())

# A. Standardise whitespace and common placeholders across text columns.
for col in clean.columns:
    clean[col] = clean[col].astype("string").str.strip()
    clean.loc[clean[col].str.upper().isin(PLACEHOLDERS), col] = pd.NA

# B. Remove exact duplicate rows and record the count.
clean = clean.drop_duplicates().copy()
duplicates_removed = rows_before - len(clean)

# C. Standardise category labels.
category_cols = ["Item", "Payment Method", "Location"]
for col in category_cols:
    if col in clean.columns:
        clean[col] = clean[col].astype("string").str.strip()

# Map common variants into canonical labels (extend if new variants are found).
category_maps = {
    "Item": {
        "coffee": "Coffee", "tea": "Tea", "cake": "Cake", "cookie": "Cookie",
        "salad": "Salad", "sandwich": "Sandwich", "smoothie": "Smoothie", "juice": "Juice"
    },
    "Payment Method": {
        "cash": "Cash", "credit card": "Credit Card", "digital wallet": "Digital Wallet"
    },
    "Location": {
        "in-store": "In-store", "instore": "In-store", "takeaway": "Takeaway"
    }
}
for col, mapping in category_maps.items():
    clean[col] = clean[col].map(
        lambda x: mapping.get(str(x).strip().casefold(), x) if pd.notna(x) else pd.NA
    )

# D. Convert numeric fields; malformed strings become NaN.
for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

# E. Parse dates. format='mixed' supports rows with different valid date formats.
clean["Transaction Date"] = pd.to_datetime(clean["Transaction Date"], errors="coerce", format="mixed")

# F. Calculate missing/invalid totals from the known transaction rule.
can_calculate_total = clean["Quantity"].notna() & clean["Price Per Unit"].notna()
missing_total = clean["Total Spent"].isna()
clean.loc[missing_total & can_calculate_total, "Total Spent"] = (
    clean.loc[missing_total & can_calculate_total, "Quantity"]
    * clean.loc[missing_total & can_calculate_total, "Price Per Unit"]
)

# G. Impute remaining numeric missing values with medians.
numeric_imputation = {}
for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    median_value = clean[col].median()
    numeric_imputation[col] = median_value
    if pd.notna(median_value):
        clean[col] = clean[col].fillna(median_value)

# H. Impute missing categories with the mode; keep a log of values used.
category_imputation = {}
for col in category_cols:
    modes = clean[col].mode(dropna=True)
    mode_value = modes.iloc[0] if not modes.empty else pd.NA
    category_imputation[col] = mode_value
    if pd.notna(mode_value):
        clean[col] = clean[col].fillna(mode_value)

# I. Do not invent IDs. Trim and preserve identifiers as strings.
clean["Transaction ID"] = clean["Transaction ID"].astype("string").str.strip()

# J. Drop rows with missing/unparseable transaction dates (cannot safely infer date).
invalid_date_rows = int(clean["Transaction Date"].isna().sum())
clean = clean.loc[clean["Transaction Date"].notna()].copy()

# K. Correct final dtypes.
clean["Transaction ID"] = clean["Transaction ID"].astype("string")
clean["Item"] = clean["Item"].astype("string")
clean["Quantity"] = clean["Quantity"].astype("Int64")  # quantities are whole-number counts
clean["Price Per Unit"] = clean["Price Per Unit"].astype("float64")
clean["Total Spent"] = clean["Total Spent"].astype("float64")
clean["Payment Method"] = clean["Payment Method"].astype("string")
clean["Location"] = clean["Location"].astype("string")
clean["Transaction Date"] = pd.to_datetime(clean["Transaction Date"])

# Reorder index after row removals.
clean = clean.reset_index(drop=True)

print(f"Exact duplicate rows removed: {duplicates_removed}")
print(f"Rows excluded for invalid/missing transaction date: {invalid_date_rows}")
print("Numeric medians used (if needed):", numeric_imputation)
print("Categorical modes used (if needed):", category_imputation)
print(f"Final shape: {clean.shape}")
display(clean.head())
display(clean.dtypes.to_frame("final_dtype"))

Exact duplicate rows removed: 0
Rows excluded for invalid/missing transaction date: 460
Numeric medians used (if needed): {'Quantity': np.float64(3.0), 'Price Per Unit': np.float64(3.0), 'Total Spent': np.float64(8.0)}
Categorical modes used (if needed): {'Item': 'Juice', 'Payment Method': 'Digital Wallet', 'Location': 'Takeaway'}
Final shape: (9540, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,Digital Wallet,Takeaway,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


,final_dtype
Transaction ID,string[python]
Item,string[python]
Quantity,Int64
Price Per Unit,float64
Total Spent,float64
Payment Method,string[python]
Location,string[python]
Transaction Date,datetime64[ns]


## 5. Outlier review after cleaning
Use the IQR method again on cleaned numeric fields. This section flags values but **does not remove or cap them**. Inspect flagged rows and use business rules before changing valid transactions.

In [6]:
def iqr_outlier_summary(data, columns):
    rows = []
    for col in columns:
        s = pd.to_numeric(data[col], errors="coerce").dropna()
        if s.empty:
            continue
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        mask = data[col].notna() & ((data[col] < lower) | (data[col] > upper))
        rows.append({
            "column": col, "Q1": q1, "Q3": q3, "IQR": iqr,
            "lower_bound": lower, "upper_bound": upper,
            "outlier_count": int(mask.sum()),
            "decision": "Retain; investigate against source/business rules"
        })
    return pd.DataFrame(rows)

outliers_after = iqr_outlier_summary(clean, ["Quantity", "Price Per Unit", "Total Spent"])
display(outliers_after)

# Optional inspection of flagged records for manual review.
for _, row in outliers_after.iterrows():
    col = row["column"]
    lower, upper = row["lower_bound"], row["upper_bound"]
    flagged = clean.loc[clean[col].notna() & ((clean[col] < lower) | (clean[col] > upper))]
    print(f"\n{col}: {len(flagged)} flagged rows")
    display(flagged.head(10))

,column,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,decision
0,Quantity,2.0,4.0,2.0,-1.0,7.0,0,Retain; investigate against source/business rules
1,Price Per Unit,2.0,4.0,2.0,-1.0,7.0,0,Retain; investigate against source/business rules
2,Total Spent,4.0,12.0,8.0,-8.0,24.0,259,Retain; investigate against source/business rules



Quantity: 0 flagged rows


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date



Price Per Unit: 0 flagged rows


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date



Total Spent: 259 flagged rows


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
10,TXN_2548360,Salad,5,5.0,25.0,Cash,Takeaway,2023-11-07
48,TXN_6342161,Salad,5,5.0,25.0,Digital Wallet,Takeaway,2023-01-08
49,TXN_8914892,Juice,5,5.0,25.0,Digital Wallet,Takeaway,2023-03-15
92,TXN_5220895,Salad,5,5.0,25.0,Cash,In-store,2023-06-10
96,TXN_9517146,Juice,5,5.0,25.0,Cash,Takeaway,2023-10-30
142,TXN_8687151,Salad,5,5.0,25.0,Cash,Takeaway,2023-06-10
149,TXN_4283157,Salad,5,5.0,25.0,Digital Wallet,In-store,2023-11-25
167,TXN_1896955,Salad,3,5.0,25.0,Digital Wallet,In-store,2023-09-23
203,TXN_8693704,Salad,3,5.0,25.0,Cash,In-store,2023-05-04
310,TXN_5523450,Salad,5,5.0,25.0,Credit Card,Takeaway,2023-07-02


## 6. Validation checks
These checks help catch common cleaning mistakes. If any check fails, inspect the relevant rows before exporting.

In [7]:
# Placeholder strings should not remain in the cleaned categorical/numeric/date fields.
placeholder_remaining = {}
for col in clean.columns:
    placeholder_remaining[col] = int(
        clean[col].astype("string").str.strip().str.upper().isin(PLACEHOLDERS).sum()
    )
print("Placeholder values remaining by column:")
display(pd.Series(placeholder_remaining, name="remaining_placeholders").to_frame())

print("Exact duplicate rows remaining:", int(clean.duplicated().sum()))
print("Missing values remaining by column:")
display(clean.isna().sum().to_frame("null_count"))

# Domain checks; these are diagnostic, not automatic deletion rules.
print("Quantity <= 0:", int((clean["Quantity"] <= 0).sum()))
print("Price Per Unit < 0:", int((clean["Price Per Unit"] < 0).sum()))
print("Total Spent < 0:", int((clean["Total Spent"] < 0).sum()))

# Check arithmetic consistency for rows with original measured values.
# Small tolerance handles floating-point representation.
expected_total = clean["Quantity"].astype(float) * clean["Price Per Unit"]
inconsistent_total = ~np.isclose(clean["Total Spent"], expected_total, rtol=0, atol=1e-6)
print("Rows where Total Spent != Quantity × Price Per Unit:", int(inconsistent_total.sum()))

Placeholder values remaining by column:


,remaining_placeholders
Transaction ID,0
Item,0
Quantity,0
Price Per Unit,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


Exact duplicate rows remaining: 0
Missing values remaining by column:


,null_count
Transaction ID,0
Item,0
Quantity,0
Price Per Unit,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


Quantity <= 0: 0
Price Per Unit < 0: 0
Total Spent < 0: 0
Rows where Total Spent != Quantity × Price Per Unit: 735


## 7. Before vs. after summary
For dtype accuracy, the expected schema is declared explicitly. The before score checks whether raw columns already match it; the after score checks the cleaned columns. Missing-value counts include true nulls after placeholders have been normalised in the cleaned data.

In [8]:
expected_dtypes = {
    "Transaction ID": "string",
    "Item": "string",
    "Quantity": "Int64",
    "Price Per Unit": "float64",
    "Total Spent": "float64",
    "Payment Method": "string",
    "Location": "string",
    "Transaction Date": "datetime64[ns]"
}

def dtype_accuracy(data, expected):
    present = [c for c in expected if c in data.columns]
    correct = sum(str(data[c].dtype) == expected[c] for c in present)
    return correct, len(expected), 100 * correct / len(expected) if expected else 100.0

# Raw missing count treats common placeholder strings as missing-like.
before_missing_like = int(sum(missing_like_mask(raw[c]).sum() for c in raw.columns))
after_nulls = int(clean.isna().sum().sum())
before_correct, before_total, before_pct = dtype_accuracy(raw, expected_dtypes)
after_correct, after_total, after_pct = dtype_accuracy(clean, expected_dtypes)

summary = pd.DataFrame([
    {
        "Metric": "Rows",
        "Before": len(raw),
        "After": len(clean),
        "Change": len(clean) - len(raw)
    },
    {
        "Metric": "Columns",
        "Before": raw.shape[1],
        "After": clean.shape[1],
        "Change": clean.shape[1] - raw.shape[1]
    },
    {
        "Metric": "Missing-like cells (null + placeholders)",
        "Before": before_missing_like,
        "After": after_nulls,
        "Change": after_nulls - before_missing_like
    },
    {
        "Metric": "Exact duplicate rows",
        "Before": int(raw.duplicated().sum()),
        "After": int(clean.duplicated().sum()),
        "Change": int(clean.duplicated().sum()) - int(raw.duplicated().sum())
    },
    {
        "Metric": "Columns matching expected dtype",
        "Before": f"{before_correct}/{before_total}",
        "After": f"{after_correct}/{after_total}",
        "Change": "—"
    },
    {
        "Metric": "Dtype accuracy (%)",
        "Before": round(before_pct, 1),
        "After": round(after_pct, 1),
        "Change": round(after_pct - before_pct, 1)
    }
])
display(summary)

,Metric,Before,After,Change
0,Rows,10000,9540,-460
1,Columns,8,8,0
2,Missing-like cells (null + placeholders),10082,0,-10082
3,Exact duplicate rows,0,0,0
4,Columns matching expected dtype,4/8,8/8,—
5,Dtype accuracy (%),50.0,100.0,50.0


## 8. Export cleaned data to a new CSV
The raw file is left unchanged. The output is saved as `cleaned_cafe_sales.csv` in the notebook's current working directory.

In [9]:
OUTPUT_PATH = Path("cleaned_cafe_sales.csv")
clean.to_csv(OUTPUT_PATH, index=False, date_format="%Y-%m-%d")

print(f"Saved cleaned dataset to: {OUTPUT_PATH.resolve()}")
print(f"Exported rows: {len(clean):,}")
print(f"Exported columns: {len(clean.columns)}")

# Re-load the exported file to verify that it is readable.
check = pd.read_csv(OUTPUT_PATH)
print("Re-loaded export shape:", check.shape)
display(check.head())

Saved cleaned dataset to: /cleaned_cafe_sales.csv
Exported rows: 9,540
Exported columns: 8


Re-loaded export shape: (9540, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,Digital Wallet,Takeaway,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## 9. Final project notes

Include these points in your project README or submission:

- The original file was preserved; cleaning was performed on a copy.
- Placeholders and invalid numeric/date strings were treated as missing-like values.
- Exact duplicate rows were removed and the number removed is printed in the notebook.
- Category formatting and data types were standardised.
- Missing numeric values used medians, missing categories used modes, and missing totals were derived from quantity × unit price when possible.
- Rows with missing/unparseable transaction dates were excluded rather than assigned a guessed date.
- IQR outliers were flagged for review and retained because unusual values are not automatically errors.
- The notebook reports before/after row counts, missingness, duplicates, and dtype accuracy.
- **Limitation:** statistical imputation can bias sales metrics. For production use, prefer correcting data from source receipts or recording explicit missing-value indicators.